# Capitolo 16 – Master theorem: chiamate e ricorrenze a confronto con la soluzione (C++)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap16_master_theorem_cpp.ipynb)

Il capitolo non contiene algoritmi con pseudocodice: questo notebook è quindi breve e **misura** quello che il Master theorem predice. Prima si contano le chiamate di `sumRecHalving`, l'algoritmo di somma per dimezzamento del codice del corso, la cui ricorrenza $T(n)=2T(n/2)+1$ cade nel caso 1 ($\Theta(n)$, in realtà esattamente $2n-1$ chiamate); poi si calcolano direttamente alcune ricorrenze $T(n)=aT(n/b)+f(n)$ degli esempi del capitolo e si verifica che il rapporto con la funzione data dal Master theorem tende a una costante.

Il capitolo completo, con pseudocodice, dimostrazioni ed esempi, è [sul sito](https://fabiofurini.github.io/algoritmi/ricorsione/16-master-theorem/).

Ogni programma è scritto in un file con la cella `%%writefile` e poi compilato ed eseguito con `g++ -std=c++17 -O2`: si eseguono le celle in ordine, dall'alto in basso. Nessuna libreria esterna.

**Indici.** Nello pseudocodice delle dispense gli array sono indicizzati da 1; nel codice (come in tutti i linguaggi qui usati) da 0: l'elemento `a[i]` dello pseudocodice è l'elemento di indice `i-1` nel codice, e un intervallo `a[l..r]` diventa `a[l-1..r-1]`. Le stampe dei notebook usano gli indici da 0.

**Nomi.** Le funzioni hanno il nome dello pseudocodice, nella convenzione del linguaggio (come nei capitoli di introduzione): `sumRecHalving` → `sumRecHalving`.

## 1. Le chiamate di `sumRecHalving` (caso 1)

`sumRecHalving`$(a,l,r)$ — *Input*: un array e due indici $l\le r$. *Output*: la somma di `a[l..r]`, ottenuta sommando le somme delle due metà. Ogni chiamata fa un lavoro costante, quindi $T(n)=2T(n/2)+1$: $a=2$, $b=2$, $n^{\log_b a}=n$, caso 1, $T(n)=\Theta(n)$. Il capitolo riporta $15$, $1\,999$ e $2\,097\,151$ chiamate per $n=8$, $1000$ e $2^{20}$, cioè esattamente $2n-1$.

Il vettore contiene $1,2,\dots,n$, quindi la somma attesa è $n(n+1)/2$.

In [1]:
%%writefile sum_rec_halving.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

int calls = 0;   // numero di chiamate

// Somma di a[l..r] (indici da 0) per dimezzamento
long long sumRecHalving(int *a, int l, int r)
{
    calls++;
    if (l == r)
    {
        return a[l];
    }
    int mid = l + (r - l) / 2;
    return sumRecHalving(a, l, mid) + sumRecHalving(a, mid + 1, r);
}

#ifndef NOMAIN
int main()
{
    int ns[] = {8, 1000, 1 << 20};
    for (int n : ns)
    {
        vector<int> a(n);
        for (int i = 0; i < n; i++)
        {
            a[i] = i + 1;
        }
        calls = 0;
        long long s = sumRecHalving(a.data(), 0, n - 1);
        printf("n=%-8d somma=%-14lld chiamate=%-9d 2n-1=%d\n", n, s, calls, 2 * n - 1);
    }
    return 0;
}
#endif

Writing sum_rec_halving.cpp


In [2]:
!g++ -std=c++17 -O2 -o sum_rec_halving sum_rec_halving.cpp && ./sum_rec_halving

n=8        somma=36             chiamate=15        2n-1=15
n=1000     somma=500500         chiamate=1999      2n-1=1999
n=1048576  somma=549756338176   chiamate=2097151   2n-1=2097151


**Prova tu.** Cambia `n` (anche un valore che non è una potenza di 2) e verifica che le chiamate siano sempre $2n-1$.

In [3]:
%%writefile prova_sum_rec_halving.cpp
#define NOMAIN
#include "sum_rec_halving.cpp"

int main()
{
    // Prova con un n che non è una potenza di 2: le chiamate sono ancora 2n-1.
    int n = 777;
    vector<int> a(n);
    for (int i = 0; i < n; i++) a[i] = i + 1;
    calls = 0;
    long long s = sumRecHalving(a.data(), 0, n - 1);
    printf("%lld %d %d\n", s, calls, 2 * n - 1);
    return 0;
}

Writing prova_sum_rec_halving.cpp


In [4]:
!g++ -std=c++17 -O2 -o prova_sum_rec_halving prova_sum_rec_halving.cpp && ./prova_sum_rec_halving

302253 1553 1553


## 2. Ricorrenze degli esempi del capitolo e Master theorem

Per $T(n)=a\,T(n/b)+f(n)$, $T(1)=1$ e $n=b^k$, la funzione `valuta` calcola $T(n)$ dalla definizione ($k$ passi: $T(b^{i})=a\,T(b^{i-1})+f(b^{i})$). Il Master theorem dà l'ordine di grandezza $\Theta(g)$ con $f(n)=n^p$ e $n^{\log_b a}=a^k$:

- **caso 1** ($f$ cresce meno): $g=n^{\log_b a}=a^k$;
- **caso 2** ($f$ e $n^{\log_b a}$ dello stesso ordine): $g=f(n)\cdot\log_b n=n^p\,k$;
- **caso 3** ($f$ cresce di più): $g=f(n)=n^p$.

L'ultima colonna è $T(n)/g(n)$: se il Master theorem è corretto, è un numero limitato e lontano da 0 (per $n$ grande si avvicina a una costante).

In [5]:
%%writefile master_theorem.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

long long IPow(long long x, int e)
{
    long long r = 1;
    for (int i = 0; i < e; i++)
    {
        r *= x;
    }
    return r;
}

// T(n) = a*T(n/b) + n^p, T(1) = 1, per n = b^k
void Valuta(int a, int b, int p, int k, long long &n, long long &t)
{
    n = 1;
    t = 1;
    for (int i = 0; i < k; i++)
    {
        n *= b;
        t = a * t + IPow(n, p);
    }
}

struct Riga
{
    const char *nome;
    int a, b, p, k, caso;
    const char *tipo;
    const char *gname;
};

#ifndef NOMAIN
int main()
{
    // (ricorrenza, a, b, p, k, caso, tipo di g, nome di g)
    vector<Riga> righe = {
        {"T(n)=T(n/2)+1",      1, 2, 0, 20, 2, "level",  "log n"},
        {"T(n)=2T(n/2)+1",     2, 2, 0, 20, 1, "leaves", "n"},
        {"T(n)=2T(n/2)+n",     2, 2, 1, 20, 2, "level",  "n log n"},
        {"T(n)=3T(n/2)+n",     3, 2, 1, 20, 1, "leaves", "n^log2(3)"},
        {"T(n)=7T(n/2)+n^2",   7, 2, 2, 20, 1, "leaves", "n^log2(7)"},
        {"T(n)=16T(n/4)+n",   16, 4, 1, 10, 1, "leaves", "n^2"},
        {"T(n)=8T(n/2)+n^3",   8, 2, 3, 15, 2, "level",  "n^3 log n"},
        {"T(n)=8T(n/3)+n^2",   8, 3, 2, 12, 3, "f",      "n^2"},
    };
    printf("%-18s caso %8s %20s  %-10s T/g\n", "ricorrenza", "n", "T(n)", "Theta(g)");
    for (const Riga &r : righe)
    {
        long long n, t;
        Valuta(r.a, r.b, r.p, r.k, n, t);
        string tipo = r.tipo;
        long long g = tipo == "leaves" ? IPow(r.a, r.k) : (tipo == "level" ? IPow(n, r.p) * r.k : IPow(n, r.p));
        printf("%-18s %4d %8lld %20lld  %-10s %.3f\n", r.nome, r.caso, n, t, r.gname, (double)t / (double)g);
    }
    return 0;
}
#endif

Writing master_theorem.cpp


In [6]:
!g++ -std=c++17 -O2 -o master_theorem master_theorem.cpp && ./master_theorem

ricorrenza         caso        n                 T(n)  Theta(g)   T/g
T(n)=T(n/2)+1         2  1048576                   21  log n      1.050
T(n)=2T(n/2)+1        1  1048576              2097151  n          2.000
T(n)=2T(n/2)+n        2  1048576             22020096  n log n    1.050
T(n)=3T(n/2)+n        1  1048576          10458256051  n^log2(3)  2.999
T(n)=7T(n/2)+n^2      1  1048576   186180488678924301  n^log2(7)  2.333
T(n)=16T(n/4)+n       1  1048576        1466015154176  n^2        1.333
T(n)=8T(n/2)+n^3      2    32768      562949953421312  n^3 log n  1.067
T(n)=8T(n/3)+n^2      3   531441        1992110014441  n^2        7.053


**Prova tu.** Cambia i parametri $a$, $b$, $p$, $k$ e confronta $T(n)$ con la funzione del caso del Master theorem.

In [7]:
%%writefile prova_master_theorem.cpp
#define NOMAIN
#include "master_theorem.cpp"

int main()
{
    // Prova una ricorrenza tua: T(n) = 4T(n/2) + n (caso 1, g = n^2) con k = 10.
    long long n, t;
    Valuta(4, 2, 1, 10, n, t);
    printf("%lld %lld %.3f\n", n, t, (double)t / (double)IPow(4, 10));
    return 0;
}

Writing prova_master_theorem.cpp


In [8]:
!g++ -std=c++17 -O2 -o prova_master_theorem prova_master_theorem.cpp && ./prova_master_theorem

1024 2096128 1.999
